# 📊 Aula 06 — Avaliação e Validação de Modelos

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Datasets:** 3 cenários (A=normal, B=overfit, C=underfit)

---

## Contexto

Você é o engenheiro de dados da planta e recebeu **3 versões** do dataset da coluna de destilação. Cada uma foi preparada para apresentar um problema diferente. Sua missão: diagnosticar cada caso com métricas de engenharia.


## 3.1 — Pipeline de Validação

Siga os 5 passos para cada cenário.


### Passo 1: Importar bibliotecas


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt


### Passo 2: Função de avaliação


In [ ]:
def avaliar_modelo(df, nome_cenario, modelo='RF'):
    print(f"\n{'='*50}")
    print(f"Cenário: {nome_cenario}  (modelo: {modelo})")

    feature_cols = [c for c in df.columns if c not in ['timestamp', 'composicao']]
    X = df[feature_cols]
    y = df['composicao']

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Escolher modelo
    if modelo == 'RF':
        clf = RandomForestRegressor(n_estimators=100, random_state=42)
        label = 'Random Forest'
    else:
        clf = LinearRegression()
        label = 'Regressão Linear'

    # Treinar
    clf.fit(X_train, y_train)

    # Predizer
    y_pred_train = clf.predict(X_train)
    y_pred_test = clf.predict(X_test)

    # Métricas
    r2_tr = r2_score(y_train, y_pred_train)
    r2_te = r2_score(y_test, y_pred_test)
    rmse_te = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae_te = mean_absolute_error(y_test, y_pred_test)

    print(f"{label}: R² treino={r2_tr:.4f}")
    print(f"R² teste:  {r2_te:.4f}")
    print(f"Diferença: {r2_tr - r2_te:.4f}")
    print(f"RMSE teste: {rmse_te:.4f}")
    print(f"MAE teste:  {mae_te:.4f}")
    print(f"RMSE/MAE: {rmse_te/mae_te:.2f} (>>1 = outliers nas predições)")

    # Validação cruzada
    cv_scores = cross_val_score(clf, X, y, cv=5, scoring='r2')
    print(f"CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    # Diagnóstico
    diff = r2_tr - r2_te
    if diff > 0.15 and r2_tr > 0.9:
        print(">>> DIAGNÓSTICO: 🔴 OVERFITTING (treino muito acima do teste)")
    elif r2_te < 0.5:
        print(">>> DIAGNÓSTICO: 🟡 UNDERFITTING (R² teste baixo)")
    else:
        print(">>> DIAGNÓSTICO: ✅ OK (modelo generaliza bem)")

    return df


### Passo 3: Executar para cada cenário


In [ ]:
BASE = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula06/"

# Cenário A: RF (deve generalizar bem → OK)
df_A = pd.read_csv(BASE + 'cenario_A.csv')
avaliar_modelo(df_A, 'A - normal', modelo='RF')

# Cenário B: RF (poucos dados + muitas features → overfitting)
df_B = pd.read_csv(BASE + 'cenario_B.csv')
avaliar_modelo(df_B, 'B - overfit', modelo='RF')

# Cenário C: relação NÃO-LINEAR → Regressão Linear (underfitting)
df_C = pd.read_csv(BASE + 'cenario_C.csv')
avaliar_modelo(df_C, 'C - underfit', modelo='linear')


### Passo 4: Predicted vs Real


In [ ]:
# Cenário A — exemplo
df_A = pd.read_csv(BASE + 'cenario_A.csv')
X_A = df_A.drop(['timestamp', 'composicao'], axis=1)
y_A = df_A['composicao']

X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(X_A, y_A, test_size=0.2, random_state=42)
rf_A = RandomForestRegressor(n_estimators=100, random_state=42)
rf_A.fit(X_train_A, y_train_A)
y_pred_A = rf_A.predict(X_test_A)

plt.figure(figsize=(6, 6))
plt.scatter(y_test_A, y_pred_A, alpha=0.3, s=10)
plt.plot([y_test_A.min(), y_test_A.max()], [y_test_A.min(), y_test_A.max()], 'r--', lw=2)
plt.xlabel('Composição real')
plt.ylabel('Composição predita')
plt.title('Cenário A — Predito vs Real')
plt.tight_layout()
plt.show()


### Passo 5: Gráfico de Resíduos


In [ ]:
residuos = y_test_A - y_pred_A
plt.figure(figsize=(10, 4))
plt.scatter(y_pred_A, residuos, alpha=0.3, s=10)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predição')
plt.ylabel('Resíduo (real - predito)')
plt.title('Gráfico de Resíduos — Cenário A')
plt.tight_layout()
plt.show()


---

## 3.2 — Exercício em Grupo: Prescrição de Melhoria

Cada grupo recebe um diagnóstico e recomenda ações corretivas.

| Grupo | Diagnóstico | Ações |
|-------|-------------|-------|
| **A** | Overfitting | Reduzir `max_depth`, aumentar dados, regularizar |
| **B** | Underfitting | Aumentar complexidade, criar features, reduzir regularização |
| **C** | OK | Testar XGBoost, features não-lineares, ensemble |
| **D** | Overfitting extremo | Mesmo que A + verificar data leakage |


In [ ]:
# Exemplo Grupo A: reduzir max_depth para combater overfitting
for depth in [3, 5, 10, None]:
    rf_test = RandomForestRegressor(n_estimators=100, max_depth=depth, random_state=42)
    rf_test.fit(X_train_A, y_train_A)
    r2_tr = r2_score(y_train_A, rf_test.predict(X_train_A))
    r2_te = r2_score(y_test_A, rf_test.predict(X_test_A))
    print(f"max_depth={depth}: R² treino={r2_tr:.4f} | R² teste={r2_te:.4f} | gap={r2_tr-r2_te:.4f}")


---

## Checklist de Validação

- [ ] Métricas calculadas (MAE, RMSE, R²)
- [ ] Diferença treino-teste
- [ ] k-fold executado
- [ ] Diagnóstico emitido
- [ ] Ação corretiva recomendada
- [ ] Resíduos plotados
